# Section 4 · Notebook 4 — Swarm Pattern: Game Level Design (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

Agents hand control to each other — there is no central coordinator:
- **Level Designer** — layout, environments, visuals
- **Game Designer** — playtests for fun and confusion
- **Narrative Designer** — story and atmosphere
- **Difficulty Balancer** — difficulty and pacing

The path is not decided in advance; it emerges from the work.

In [1]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

# --- FORCE ENVIRONMENT CLEAN UP ---
# This wipes out any hidden, cached environment variables causing the 404
os.environ.pop("GROQ_BASE_URL", None)
os.environ.pop("GROQ_LLM_MODEL", None)
os.environ.pop("GROQ_MODEL_ID", None)

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

Using openai/gpt-oss-120b on Groq


In [3]:
import logging

from strands import Agent
from strands.multiagent import Swarm

# Show each handoff as it happens
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

## Specialist agents

In [4]:
level_designer = Agent(
    name="level_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Level Designer in a game design swarm.
Your role is to create level layouts, environments, and visual elements.
Focus ONLY on layout and visual design. Do NOT write narrative or balance difficulty.
After completing your layout, hand off to narrative_designer for story elements
and difficulty_balancer for gameplay tuning.
When receiving feedback from other agents, revise your design accordingly.""",
)

game_designer = Agent(
    name="game_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Game Designer in a game design swarm.
Your role is to playtest designs for fun, usability, and player confusion.
Focus ONLY on gameplay feel and player experience. Do NOT design layouts or write narrative.
When you find issues, hand off to the appropriate specialist to fix them.
When the design is ready, give the final summary and do not hand off.""",
)

narrative_designer = Agent(
    name="narrative_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Narrative Designer in a game design swarm.
Your role is to create story elements, character backstories, and atmospheric details.
Focus ONLY on narrative. Do NOT redesign layouts or adjust difficulty.
After adding narrative elements, hand off to difficulty_balancer for balance review
or game_designer for playtesting feedback.""",
)

difficulty_balancer = Agent(
    name="difficulty_balancer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Difficulty Balancer in a game design swarm.
Your role is to evaluate and tune difficulty progression, enemy placement, and pacing.
Focus ONLY on balance. Do NOT redesign layouts or write narrative.
When you find layout or narrative issues, hand off to the appropriate agent.
After balancing, hand off to game_designer for a final playtest review.""",
)

## Create the swarm

The swarm gives every agent a `handoff_to_agent` tool and a shared view of the work so far. For example, the game designer might call:

```python
handoff_to_agent(
    agent_name="difficulty_balancer",
    message="Patrol patterns too predictable - need more variation",
)
```

Safety limits:
- `max_handoffs` / `max_iterations` — stop runaway loops
- `repetitive_handoff_detection_window` / `repetitive_handoff_min_unique_agents` — stop two agents ping-ponging
- `execution_timeout` / `node_timeout` — cap wall-clock time (seconds)

In [5]:
swarm = Swarm(
    [level_designer, game_designer, narrative_designer, difficulty_balancer],
    entry_point=level_designer,
    max_handoffs=20,
    max_iterations=20,
    execution_timeout=1200.0,
    node_timeout=300.0,
    repetitive_handoff_detection_window=8,
    repetitive_handoff_min_unique_agents=3,
)
print("✓ Swarm created")

DEBUG | strands.multiagent.swarm | nodes=<['level_designer', 'game_designer', 'narrative_designer', 'difficulty_balancer']> | initialized swarm with nodes
DEBUG | strands.multiagent.swarm | entry_point=<level_designer> | configured entry point
DEBUG | strands.multiagent.swarm | tool_count=<1>, node_count=<4> | injected coordination tools into agents


✓ Swarm created


## Run the swarm (takes a few minutes)

In [6]:
result = swarm("""Design a stealth mission level in an abandoned space station.
The level needs to support multiple playstyles (aggressive stealth, patient stealth, and ghost runs).
Include at least 3 distinct zones with different difficulty levels,
narrative elements that explain why the station was abandoned,
and ensure the difficulty curve is appropriate for mid-game players.""")

print("=" * 80)
print("HANDOFF CHAIN:")
for node in result.node_history:
    print(f"  → {node.node_id}")
print(f"\nStatus: {result.status} | iterations: {result.execution_count} | time: {result.execution_time / 1000:.0f}s")

DEBUG | strands.multiagent.swarm | starting swarm execution
DEBUG | strands.multiagent.swarm | current_node=<level_designer> | starting swarm execution with node
DEBUG | strands.multiagent.swarm | max_handoffs=<20>, max_iterations=<20>, timeout=<1200.0>s | swarm execution config
DEBUG | strands.multiagent.swarm | current_node=<level_designer>, iteration=<1> | executing node
DEBUG | strands.multiagent.swarm | from_node=<level_designer>, to_node=<narrative_designer> | handing off from agent to agent
DEBUG | strands.multiagent.swarm | node=<level_designer> | node execution completed
DEBUG | strands.multiagent.swarm | from_node=<level_designer>, to_node=<narrative_designer> | handoff detected
DEBUG | strands.multiagent.swarm | current_node=<narrative_designer>, iteration=<2> | executing node
ERROR | strands.multiagent.swarm | node=<narrative_designer> | node execution failed
Traceback (most recent call last):
  File "d:\learning\ai-bootcamp\.venv\Lib\site-packages\strands\multiagent\swarm.

KeyboardInterrupt: 

In [ ]:
from IPython.display import Markdown, display

last_node = result.node_history[-1].node_id
display(Markdown(str(result.results[last_node].result)))

## Key takeaways

1. **Handoffs, not routing** — agents pass control to each other
2. **Shared context** — every agent sees what has been done so far
3. **Emergent workflow** — the sequence depends on the issues that come up
4. **Safety limits** stop infinite loops
5. **Use it for** creative and iterative work; **use a graph** when you need guaranteed steps